# 12 - Train a Reasoning DQN Model Offline

Same offline loop as `02_train_offline_dqn.ipynb`, with Coconut-style **latent reasoning** added:

1. Every step ends with a learned **action-prompt token**: a trailing `learnable` modality named `value` in both the tokenizer and the embedder, flagged `head_output: True`. Q-values are read from the head-output token(s); with `tokens > 1` each step yields several Q rows, all trained toward the same TD target.
2. Each training batch samples one **burst step** per sequence (`sample_reasoning_splits`). The model generates `NUM_THOUGHTS` latent "thought" embeddings there, on the autograd tape: each thought's input is the `LatentReasoner` adapter applied to the backbone's output at the previous position.
3. The thoughts are inserted **between the burst step's data tokens and its action prompt**, so the prompt (and every later token in the same task) attends to them.
4. The unchanged `DqnObjective` TD loss backpropagates through the latent chain, training the backbone to emit thoughts that make value predictions more accurate.

Latent generation runs `NUM_THOUGHTS + 1` backbone passes per batch instead of one, so each optimizer step costs roughly `(R + 1)x` the plain forward.

This is a short usage example, not a full experiment. Evaluate a saved checkpoint in `09_inference.ipynb`.


In [ ]:
import numpy as np
import torch

from mouse_core import AdamW
from mouse_core.data import (
    DataLoader,
    Augmenter,
    Tokenizer,
    compose,
    load_stores_from_hub,
)
from mouse_core.objectives import DqnObjective
from mouse_core.models import (
    LatentReasoner,
    Model,
    Polyak,
    push_model_to_hub,
    sample_reasoning_splits,
)
from mouse_core.models.backbone import Qwen3Backbone
from mouse_core.models.embedding import NumericEmbedder
from mouse_core.models.heads import DiscreteActionValueHead


DATASET_ID = "mouse-example-dataset"          # Hugging Face dataset repo for load_stores_from_hub
MODEL_ID = "mouse-example-model-reasoning"    # Hugging Face model repo for push_model_to_hub
MAX_ACTIONS = 4                               # number of discrete actions predicted by the head
MAX_OBS_DISCRETE = 64                         # vocabulary size for discrete observations
SEQUENCE_LENGTH = 512                         # replay sequence length sampled by DataLoader
BATCH_SIZE = 4                                # sequences per optimizer step
NUM_CYCLES = 2                               # outer train cycles (print cadence)
TRAIN_STEPS = 50                             # optimizer updates per cycle (passed to run_train)
POLYAK_TAU_HEADS = 0.0001                     # delayed Q-head interpolation (0 = frozen, 1 = copy of the online heads)
POLYAK_TAU_ENCODER = 0.01                     # delayed encoder interpolation
POLYAK_TAU_BACKBONE = 0.01                    # delayed backbone interpolation
NUM_THOUGHTS = 4                              # latent thoughts generated per reasoning burst


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


## Load Data

`load_stores_from_hub` downloads the dataset snapshot and reconstructs the saved `Datastore` objects. Each returned store is one ordered environment stream.


In [ ]:
stores = load_stores_from_hub(repo_id=DATASET_ID, split='train', force_download=True)

## Data pipeline

`DataLoader` samples contiguous windows up to `sequence_length` (a max) from one or more datastores. Each sequence may be shorter than the max depending on where the window starts in the store.

Pipeline order: `augmenter → tokenizer → pack → embedder`.

| Stage | Role |
| --- | --- |
| **Augmenter** | `dict → dict` (`fields=` value transforms; `seed_field=` for shared draws within a `reseed` generation). Action permute sets `input_vector_field` / `output_vector_field` on `info_q_star` so Q* stays aligned. |
| **Tokenizer** | `dict → StepTokens` (`input_field` / `output_field`; `objective_fields=` is `action` / `reward` / `episode_done` / `task_done`; `grouping_field=`) |

Compose `train_transform = compose(augmenter, tokenizer)`.
`DataLoader(transform=train_transform)` maps each step and packs into a `TokenBatch`.
Live inference in `09_inference.ipynb` uses the tokenizer without the augmenter so chosen actions match the env.

Same trailing `learnable` `value` token as `02` (the head-output / action prompt). The addition here is latent reasoning: at the burst step the thoughts slot in right before that token, `[data tokens][thought 1..R][value]`.


In [ ]:
# Pipeline order: augmenter → tokenizer

augmenter = Augmenter(
    seed_field="task_index",
    fields=[
        {
            "type": "discrete",
            "input_field": "action",
            "input_vector_field": "info_q_star",
            "vocab_size": MAX_ACTIONS,
            "permute": True,
        },
        {
            "type": "discrete",
            "input_field": "observation",
            "vocab_size": MAX_OBS_DISCRETE,
            "permute": True,
        },
    ],
)

tokenizer = Tokenizer(
    input_fields=[
        {
            "type": "discrete",
            "input_field": "action",
        },
        {
            "type": "discrete",
            "input_field": "observation",
        },
        {
            "type": "fourier",
            "input_field": "reward",
        },
        {
            "type": "discrete",
            "input_field": "episode_done",
        },
        # Action prompt: a trailing learned token per step, named
        # "value" and flagged as the head-output field -- Q is read
        # from its token(s). With tokens > 1 every step yields several Q
        # rows, all trained toward the same TD target; reasoning latents
        # are inserted immediately before the step's first head-output
        # token, so every head-output token attends to them.
        {
            "type": "learnable",
            "output_field": "value",
            "tokens": 1,
            "head_output": True,
        },
    ],
    objective_fields=[
        {
            "input_field": "action",
        },
        {
            "input_field": "reward",
        },
        {
            "input_field": "episode_done",
        },
        {
            "input_field": "task_done",
        },
    ],
    grouping_field="task_index",
)

train_transform = compose(augmenter, tokenizer)

loader = DataLoader(
    stores=stores,
    sequence_length=SEQUENCE_LENGTH,
    batch_size=BATCH_SIZE,
    transform=train_transform,
    prefetch=4,
    num_workers=0,
)


## Build The Model

A Mouse Core `Model` has three main pieces, plus the optional reasoner:

- `NumericEmbedder` maps a tokenized `TokenBatch` (modalities keyed by name; add `vocab_size` / `std` here; `fourier` / `continuous` also need `fourier_min` / `fourier_max`) into vectors. The trailing `learnable` modality named `value` embeds the action-prompt token.
- `Qwen3Backbone` processes those tokens with a transformer backbone.
- `DiscreteActionValueHead` predicts one value per discrete action, read from each step's prompt token.
- `LatentReasoner` is the thought adapter (LayerNorm + Linear): it maps the backbone's output hidden state at the previous position to the input embedding of the next latent thought. `num_thoughts` fixes the burst length `R`.

The backbone exposes `hidden_dim`, and the embedder, head, and reasoner use that same value so the pieces connect cleanly.

`NumericEmbedder` modality types used here:

- `discrete` for integer IDs such as actions, observations, and episode/task done codes.
- `fourier` for scalar numeric values such as rewards.
- `learnable` for the action-prompt token (learned tokens not tied to a row field; named `value` here via `field=` / `output_field=`).

`Model(...)` wraps the pieces behind a single forward call that returns predictions, objective data, and an optional cache. The reasoner is saved and loaded with the checkpoint.


In [ ]:
backbone = Qwen3Backbone(
    train_kernel="flex",
    decode_kernel="flex",
    dtype=torch.float32,
    pretrained="Qwen/Qwen3-0.6B",
)

encoder = NumericEmbedder(
    hidden_dim=backbone.hidden_dim,
    modalities=[
        {
            "type": "discrete",
            "field": "action",
            "vocab_size": MAX_ACTIONS,
            "std": 0.02,
            "positions": 1,
        },
        {
            "type": "discrete",
            "field": "observation",
            "vocab_size": MAX_OBS_DISCRETE,
            "std": 0.02,
            "positions": 1,
        },
        {
            "type": "fourier",
            "field": "reward",
            "std": 0.02,
            "positions": 1,
            "fourier_min": 0.01,
            "fourier_max": 10.0,
        },
        {
            "type": "discrete",
            "field": "episode_done",
            "vocab_size": 3,
            "std": 0.02,
            "positions": 1,
        },
        # Action prompt (name matches the tokenizer's output_field).
        {
            "type": "learnable",
            "field": "value",
            "tokens": 1,
            "std": 0.02,
            "positions": 1,
        },
    ],
)

head = DiscreteActionValueHead(
    in_features=backbone.hidden_dim,
    out_features=MAX_ACTIONS,
    hidden_dim=backbone.hidden_dim,
    num_layers=1,
    scale=0.1,
)

reasoner = LatentReasoner(hidden_dim=backbone.hidden_dim, num_thoughts=NUM_THOUGHTS)

model = Model(
    encoder=encoder,
    backbone=backbone,
    heads=head,
    action_head="action_value",
    reasoner=reasoner,
    recurrence=None,
).train().to(device)
print(model)


## Training Phase

Each outer cycle runs `TRAIN_STEPS` optimizer updates via `run_train`. Mouse Core abstractions do most of the work:

1. `inputs, objective_data = loader.next_batch()` samples ragged step windows (up to `SEQUENCE_LENGTH`).
2. `sample_reasoning_splits(inputs, rng)` picks one burst step per sequence -- uniform over steps whose next step shares the task grouping, so the TD pair out of the burst carries loss weight.
3. `model(inputs, reasoning=splits)` embeds the `TokenBatch`, generates `NUM_THOUGHTS` latent thoughts per burst on the autograd tape (`R` extra backbone passes over the growing prefixes), inserts them before the burst step's first head-output token, and runs the final pass over the extended stream. Predictions keep the flat one-row-per-head-output-token shape, so the objective is unchanged.
4. `objective(objective_data, predictions, delayed_predictions)` computes the DQN loss and metrics. TD errors at the burst step and every later same-run step backpropagate through the latent chain.
5. `AdamW` updates weights. The backbone, encoder, and heads are fp32 (`dtype=torch.float32`), so every update lands in fp32 with no master weights.
6. Delayed Q comes from the delayed model: `delayed_model = model.delayed_copy()` is a frozen copy of the online model — the fp32 encoder, backbone, reasoner adapter, and Q head are copied. `delayed_model(inputs, reasoning=splits)` runs the same `TokenBatch` with the same burst steps, so the delayed model generates its own latent thoughts through its delayed weights. `torch.no_grad()` keeps the target off the tape. `polyak.update(tau_heads=POLYAK_TAU_HEADS, tau_encoder=POLYAK_TAU_ENCODER, tau_backbone=POLYAK_TAU_BACKBONE)` interpolates each section toward the online model after the optimizer step: `0` keeps it frozen, `1` copies the online weights (no delay). Every interpolated parameter is fp32, so a small `tau` is never rounded away.

`DqnObjective` interprets `episode_done` and `task_done` (each `0`/`1`/`2`) through separate discount factors. The bootstrap is multiplied by the episode gamma, then by the task gamma (`1.0` when `task_done` is `0`). When a task ends both fire, so a task gamma of `0.0` zeros the whole term.


In [ ]:
optimizer = AdamW(
    model.parameters(),
    lr=1e-05,
    weight_decay=0.0,
    betas=(0.9, 0.95),
    eps=1e-08,
)
delayed_model = model.delayed_copy()
polyak = Polyak(model, delayed_model)
objective = DqnObjective(
    gamma_step=1.0,
    gamma_episode_terminal=1.0,
    gamma_episode_truncated=1.0,
    gamma_task_terminal=0.0,
    gamma_task_truncated=0.0,
    grouping_field="task_index",
)
rng = np.random.default_rng(0)

def run_train(*, model: Model, delayed_model: Model, polyak: Polyak, optimizer: AdamW, objective: DqnObjective, loader: DataLoader, num_steps: int) -> tuple[torch.Tensor, dict[str, float]]:
    """Run ``num_steps`` optimizer steps on batches from ``loader``."""
    model.train()
    loss: torch.Tensor | None = None
    metrics: dict[str, float] = {}
    for _ in range(num_steps):
        inputs, objective_data = loader.next_batch()
        splits = sample_reasoning_splits(inputs, rng)
        out = model(inputs, reasoning=splits)
        with torch.no_grad():
            delayed_out = delayed_model(inputs, reasoning=splits)
        loss, metrics = objective(
            objective_data.to(device),
            out.predictions,
            delayed_out.predictions,
        )
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        polyak.update(
            tau_heads=POLYAK_TAU_HEADS,
            tau_encoder=POLYAK_TAU_ENCODER,
            tau_backbone=POLYAK_TAU_BACKBONE,
        )
    assert loss is not None
    return (loss, metrics)


## Run

Each of `NUM_CYCLES` cycles calls `run_train(num_steps=TRAIN_STEPS)`. Score the checkpoint later in `09_inference.ipynb`.


In [ ]:
for cycle in range(NUM_CYCLES):
    loss, metrics = run_train(
        model=model,
        delayed_model=delayed_model,
        polyak=polyak,
        optimizer=optimizer,
        objective=objective,
        loader=loader,
        num_steps=TRAIN_STEPS,
    )
    print(f"cycle={cycle} train  loss={loss.item():.4f}  q={metrics['q_values_mean']:.3f}")
loader.close()


## Push To The Hub

`push_model_to_hub` saves the model architecture and weights together. Later, `load_model` can reconstruct the full `Model` without repeating the embedder, backbone, and head definitions.


In [ ]:
model.eval().to("cpu")
url = push_model_to_hub(model=model, repo_id=MODEL_ID, private=False, clear=True)
print(f"Pushed to {url}")